<a href="https://colab.research.google.com/github/andrew8gmf/pucrio-mvp-qssi/blob/main/puc_rio_dass42_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Engenharia de Sistemas de Software Inteligentes
## Trabalho de Machine Learning: Predição de Níveis de Depressão

**Contexto:**
Este notebook apresenta a resolução de um problema de classificação utilizando o dataset DASS-42 (Depression Anxiety Stress Scales), coletado pelo OpenPsychometrics.org.

**O Problema:**
O objetivo é classificar o nível de depressão de um indivíduo (Normal, Leve, Moderado, Grave ou Extremamente Grave) com base em seus traços de personalidade (Escala TIPI - Ten Item Personality Inventory) e dados demográficos básicos.

**Etapas contempladas:**
1. Carga e limpeza de dados (Data Quality).
2. Engenharia de features e definição do target.
3. Separação de dados (Holdout).
4. Transformação de dados (Normalização e Padronização via Pipelines).
5. Modelagem com múltiplos algoritmos (KNN, Árvore, Naive Bayes e SVM).
6. Otimização de hiperparâmetros (GridSearchCV).
7. Avaliação e comparação de resultados.
8. Exportação do modelo.

In [23]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Algoritmos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

### 1. Carga e Limpeza de Dados
Nesta etapa, aplicamos filtros de qualidade de dados (Data Quality). O dataset contém perguntas de validação (VCL) que identificam se o participante leu as instruções. Também removemos outliers de idade e tamanho de família.

In [24]:
# Definindo a URL do arquivo raw
url = 'https://raw.githubusercontent.com/andrew8gmf/pucrio-mvp-qssi/main/archive/data.csv'

# Carregando os dados (separador por tabulação conforme original)
df = pd.read_csv(url, sep='\t')

# Limpeza baseada em validade (VCL6, 9 e 12 são palavras fictícias)
df_clean = df[(df['VCL6'] == 0) & (df['VCL9'] == 0) & (df['VCL12'] == 0)].copy()

# Filtro de idade (removendo entradas irreais)
df_clean = df_clean[(df_clean['age'] >= 10) & (df_clean['age'] <= 90)]

print(f"Total de registros após limpeza: {len(df_clean)}")

Total de registros após limpeza: 34575


### 2. Definição do Alvo (Target) e Seleção de Features
O score de depressão é calculado pela soma de 14 itens específicos da escala DASS-42. O resultado é então categorizado em níveis de severidade.

As **features** escolhidas para o modelo são os 10 itens de personalidade (TIPI) e dados demográficos como educação, gênero e idade.

In [25]:
# Colunas de depressão
dep_items = ['Q3A', 'Q5A', 'Q10A', 'Q13A', 'Q16A', 'Q17A', 'Q21A', 'Q24A', 'Q26A', 'Q31A', 'Q34A', 'Q37A', 'Q38A', 'Q42A']

# Cálculo do score (normalizando de 1-4 para 0-3)
raw_dep_score = df_clean[dep_items].sum(axis=1) - 14

# Função de categorização oficial DASS-42
def classify_dep(score):
    if score <= 9: return 0  # Normal
    if score <= 13: return 1 # Leve
    if score <= 20: return 2 # Moderado
    if score <= 27: return 3 # Grave
    return 4                 # Extremamente Grave

df_clean['target'] = raw_dep_score.apply(classify_dep)

# Seleção de Features
features_tipi = [f'TIPI{i}' for i in range(1, 11)]
features_demo = ['education', 'urban', 'gender', 'age', 'married']
X = df_clean[features_tipi + features_demo]
y = df_clean['target']

### 3. Separação de Dados (Holdout)
Dividimos os dados em conjunto de treino (80%) e teste (20%), utilizando o parâmetro `stratify` para manter a proporção das classes original.

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Treino: {X_train.shape[0]} amostras, Teste: {X_test.shape[0]} amostras")

Treino: 27660 amostras, Teste: 6915 amostras


### 4. Transformação de Dados e Pipelines
Criamos um `ColumnTransformer` para tratar tipos de dados diferentes:
- **Numéricos (TIPI e Idade):** Aplicamos `StandardScaler` (Padronização).
- **Categóricos:** Aplicamos `OneHotEncoder`.

O uso de `Pipeline` é fundamental para evitar o vazamento de dados durante a validação cruzada.

In [27]:
numeric_features = features_tipi + ['age']
categorical_features = ['education', 'urban', 'gender', 'married']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

### 5. Modelagem e Comparação (Cross-Validation)
Abaixo, treinamos e comparamos os modelos solicitados utilizando Validação Cruzada (3 folds).

In [28]:
models = {
    'KNN': KNeighborsClassifier(),
    'Arvore': DecisionTreeClassifier(max_depth=10, random_state=42),
    'NaiveBayes': GaussianNB(),
    'SVM': SVC(kernel='linear') # Linear para otimizar tempo de treino em dataset grande
}

results = {}

for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])
    # Reduzindo o número de amostras para o SVM no Colab se necessário,
    # mas aqui usaremos o dataset completo com CV simplificado.
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=3)
    results[name] = cv_scores.mean()
    print(f"{name}: Acurácia Média = {cv_scores.mean():.4f}")

KNN: Acurácia Média = 0.3751
Arvore: Acurácia Média = 0.4261
NaiveBayes: Acurácia Média = 0.3984
SVM: Acurácia Média = 0.4559


### 6. Otimização de Hiperparâmetros
Utilizaremos o `GridSearchCV` para encontrar os melhores parâmetros para a Árvore de Decisão.

In [29]:
param_grid = {
    'clf__max_depth': [5, 10, 20],
    'clf__criterion': ['gini', 'entropy']
}

dt_pipe = Pipeline([('pre', preprocessor), ('clf', DecisionTreeClassifier(random_state=42))])
grid_search = GridSearchCV(dt_pipe, param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"Melhores parâmetros: {grid_search.best_params_}")
best_model = grid_search.best_estimator_

Melhores parâmetros: {'clf__criterion': 'entropy', 'clf__max_depth': 5}


### 7. Avaliação Final
Avaliação do melhor modelo no conjunto de teste (Holdout) que não foi visto durante o treinamento ou otimização.

In [30]:
y_pred = best_model.predict(X_test)
print("Relatório de Classificação Final:")
print(classification_report(y_test, y_pred))
print(f"Acurácia Final: {accuracy_score(y_test, y_pred):.4f}")

Relatório de Classificação Final:
              precision    recall  f1-score   support

           0       0.52      0.60      0.55      1546
           1       0.00      0.00      0.00       670
           2       0.25      0.22      0.23      1246
           3       0.00      0.00      0.00      1137
           4       0.48      0.83      0.61      2316

    accuracy                           0.45      6915
   macro avg       0.25      0.33      0.28      6915
weighted avg       0.32      0.45      0.37      6915

Acurácia Final: 0.4518


### 8. Exportação do Modelo
Por fim, salvamos o modelo treinado e o pipeline de pré-processamento em um arquivo `.pkl` para uso futuro em produção.

In [31]:
joblib.dump(best_model, 'data.pkl')
print("Modelo exportado com sucesso como 'modelo_dass42_final.pkl'")

Modelo exportado com sucesso como 'modelo_dass42_final.pkl'
